# EEG Data Preprocessing and Wrangling for Seizure Prediction

## Project Context
This notebook prepares EEG recordings from the CHB-MIT and Siena datasets for downstream seizure prediction modeling.
At this milestone, one EDF file per dataset is used to validate preprocessing and feature-engineering choices before scaling to all available recordings.

## Section 1: Preprocessing
Objective: standardize heterogeneous EEG recordings into a comparable signal representation suitable for cross-dataset learning.

### 1.1 Environment Setup, Imports, and Data Paths
This step installs and imports the core scientific Python dependencies required for EEG signal processing, feature extraction, and visualization.
It then defines the EDF file paths used throughout the preprocessing workflow.

In [4]:
# Run once per environment to install required packages for this notebook
%pip install -q numpy pandas scipy mne seaborn matplotlib

In [5]:
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

mne.set_log_level("WARNING")

# File paths
chb_path = "data/chbmit/chb01/chb01_01.edf"
siena_path = "data/siena/PN00/PN00-1.edf"

### 1.2 Raw EEG Loading and Initial Audit
The EDF recordings are loaded and inspected for sampling rate and channel count.
This initial audit establishes cross-dataset differences that must be harmonized before reliable seizure-related feature comparison.

In [6]:
raw_chb = mne.io.read_raw_edf(chb_path, preload=False, verbose=False)
raw_siena = mne.io.read_raw_edf(siena_path, preload=False, verbose=False)

print("CHB sampling rate:", raw_chb.info["sfreq"])
print("CHB number of channels:", len(raw_chb.ch_names))

print("Siena sampling rate:", raw_siena.info["sfreq"])
print("Siena number of channels:", len(raw_siena.ch_names))

FileNotFoundError: File does not exist: "/content/data/chbmit/chb01/chb01_01.edf"

### 1.3 Preprocessing Methodology
The preprocessing pipeline removes non-EEG channels, resolves MNE-generated duplicate channel labels, and harmonizes sampling rates across datasets.
A common target sampling frequency of 256 Hz is used to align CHB-MIT and Siena representations for cross-dataset feature comparability.
A band-pass filter (0.5-40 Hz) and 60 Hz notch filter are then applied to reduce artifact and line-noise contamination while preserving clinically relevant EEG rhythms.

In [ ]:
def drop_mne_duplicates(raw):
    to_drop = [ch for ch in raw.ch_names if ch.endswith("-1")]
    if to_drop:
        raw.drop_channels(to_drop)


def preprocess_eeg(raw, dataset_name, target_sfreq=256):
    raw = raw.copy()
    raw.load_data()

    # Drop obvious non-EEG channels
    non_eeg_keywords = [
        "EKG",
        "ECG",
        "SPO2",
        "HR",
        "PHOTIC",
        "IBI",
        "BURSTS",
        "SUPPR",
        "MK",
    ]
    drop_now = []

    for ch in raw.ch_names:
        ch_upper = ch.upper()
        if any(word in ch_upper for word in non_eeg_keywords):
            drop_now.append(ch)

    if drop_now:
        raw.drop_channels(drop_now)

    # Drop duplicate channels added by MNE
    drop_mne_duplicates(raw)

    # Resample to a common sampling rate
    if raw.info["sfreq"] != target_sfreq:
        raw.resample(target_sfreq)

    # Set all remaining channels to EEG
    raw.set_channel_types({ch: "eeg" for ch in raw.ch_names})

    # Re-reference Siena only
    if dataset_name.lower() == "siena":
        raw.set_eeg_reference("average", projection=False)

    # Bandpass filter
    raw.filter(l_freq=0.5, h_freq=40.0, verbose=False)

    # Notch filter
    raw.notch_filter(freqs=60, verbose=False)

    return raw

### 1.4 Pipeline Execution and Verification
The preprocessing function is applied to both datasets, and resulting metadata are checked to confirm expected harmonization.
This verification step supports reproducible feature extraction and cross-dataset modeling consistency.

In [ ]:
raw_chb_clean = preprocess_eeg(raw_chb, "CHB")
raw_siena_clean = preprocess_eeg(raw_siena, "Siena")

print("CHB cleaned sampling rate:", raw_chb_clean.info["sfreq"])
print("CHB cleaned channels:", len(raw_chb_clean.ch_names))

print("Siena cleaned sampling rate:", raw_siena_clean.info["sfreq"])
print("Siena cleaned channels:", len(raw_siena_clean.ch_names))

### 1.5 Epoch Construction via Sliding Windows
Continuous EEG is partitioned into overlapping fixed-length windows to create model-ready analysis units.
The current prototype uses 2-second windows with 1-second overlap, balancing temporal resolution against feature stability.
Overlapping windows improve temporal coverage of transient seizure dynamics while preserving local context.

In [ ]:
# Prototype windowing parameters: 2-second windows with 1-second overlap
def make_epochs(raw, duration=2.0, overlap=1.0):
    return mne.make_fixed_length_epochs(
        raw, duration=duration, overlap=overlap, preload=True, verbose=False
    )


epochs_chb = make_epochs(raw_chb_clean, duration=2.0, overlap=1.0)
epochs_siena = make_epochs(raw_siena_clean, duration=2.0, overlap=1.0)

X_chb = epochs_chb.get_data()
X_siena = epochs_siena.get_data()

print("CHB epoch shape:", X_chb.shape)
print("Siena epoch shape:", X_siena.shape)

### 1.6 Time-Domain Feature Extraction
Each epoch is summarized with statistical and energy-based descriptors to capture amplitude dispersion, volatility, and power concentration.
These compact representations are designed to support seizure prediction models that operate on tabular features.

In [ ]:
def extract_features(epoch_data, dataset_name, sfreq, duration=2.0):
    rows = []

    for i, epoch in enumerate(epoch_data):
        flat = epoch.flatten()

        row = {
            "dataset": dataset_name,
            "epoch_index": i,
            "epoch_start_sec": i * (duration - 1.0),
            "mean": np.mean(flat),
            "std": np.std(flat),
            "min": np.min(flat),
            "max": np.max(flat),
            "range": np.max(flat) - np.min(flat),
            "energy": np.sum(flat**2),
            "rms": np.sqrt(np.mean(flat**2)),
            "abs_mean": np.mean(np.abs(flat)),
            "channel_count": epoch.shape[0],
            "samples_per_epoch": epoch.shape[1],
            "sampling_rate": sfreq,
        }

        rows.append(row)

    return pd.DataFrame(rows)

### 1.7 Unified Feature Table Assembly
Feature matrices from CHB-MIT and Siena are concatenated into a single analysis table.
A combined table enables shared quality control, wrangling operations, and comparative exploratory analysis.

In [ ]:
df_chb = extract_features(X_chb, "CHB", raw_chb_clean.info["sfreq"], duration=2.0)
df_siena = extract_features(
    X_siena, "Siena", raw_siena_clean.info["sfreq"], duration=2.0
)

df_all = pd.concat([df_chb, df_siena], ignore_index=True)

print("Final feature table shape:", df_all.shape)
df_before = df_all.copy()
df_all

## Section 2: Data Wrangling
Objective: improve statistical robustness and feature quality before downstream seizure prediction modeling, while retaining clinically meaningful signal variation.

### 2.1 Missing-Value Assessment
Feature completeness is evaluated to ensure that downstream transformations and model fitting are not biased by absent observations.

In [ ]:
# Missing Values
print(df_all.isnull().sum())

### 2.2 Data Type Standardization
Categorical and integer casting is applied to key identifiers to support stable grouping, indexing, and reproducible analysis semantics.
This also prepares metadata fields for downstream model pipelines and evaluation reports.

In [ ]:
print(df_all.dtypes)
df_all["dataset"] = df_all["dataset"].astype("category")
df_all["epoch_index"] = df_all["epoch_index"].astype(int)

### 2.3 Outlier Diagnostics with IQR and Z-Score
Outlier profiles are quantified using two complementary methods: IQR thresholds and Z-score deviation.
Joint inspection helps distinguish distribution-tail behavior from extreme standardized departures in core predictors.

In [ ]:
from scipy.stats import zscore

cols = ["mean", "std", "range", "energy", "rms"]

for col in cols:
    Q1 = df_all[col].quantile(0.25)
    Q3 = df_all[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(col, "IQR outliers:", ((df_all[col] < lower) | (df_all[col] > upper)).sum())

    z_vals = np.abs(zscore(df_all[col]))
    print(col, "Z-score outliers:", (z_vals > 3).sum())

### 2.4 Robust Outlier Handling via IQR Clipping
Rather than removing epochs, feature values are winsorized to IQR bounds.
This preserves sample volume while reducing the influence of extreme amplitudes that may reflect artifacts rather than physiologic dynamics.

In [ ]:
for col in cols:
    Q1 = df_all[col].quantile(0.25)
    Q3 = df_all[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_all[col] = df_all[col].clip(lower, upper)

### 2.5 Derived Feature Engineering
Two additional predictors are introduced: range-to-standard-deviation ratio and per-channel energy normalization.
These derived terms improve comparability across epochs and channel configurations and can enhance seizure-related separability.

In [ ]:
df_all["range_to_std"] = df_all["range"] / (df_all["std"] + 1e-8)
df_all["energy_per_channel"] = df_all["energy"] / df_all["channel_count"]

### 2.6 Post-Wrangling Validation
Final dimensionality and missing-value checks verify that wrangling preserved table integrity while enriching the feature set.

In [ ]:
print("Before shape:", df_before.shape)
print("After shape:", df_all.shape)
print("Missing after:", df_all.isnull().sum().sum())
df_all

## Section 3: Exploratory Data Analysis (EDA)
Objective: evaluate feature behavior, dataset comparability, and early evidence of separable structure relevant to seizure prediction.
Interpretive emphasis is placed on distribution shape, feature interactions, and temporal behavior across epochs.

### 3.1 Distribution of Standard Deviation
This figure characterizes variability intensity across epochs. Heavy tails or multimodality may indicate intermittent high-variance states relevant to seizure activity.

In [ ]:
sns.histplot(df_all["std"], kde=True)
plt.title("Figure 1: Distribution of Standard Deviation")
plt.show()

### 3.2 Distribution of Energy
Energy summarizes aggregate signal power per epoch. Elevated tails can reflect transient high-amplitude dynamics often associated with abnormal neural synchronization.

In [ ]:
sns.histplot(df_all["energy"], kde=True)
plt.title("Figure 2: Distribution of Energy")
plt.show()

### 3.3 Distribution of RMS
RMS provides an amplitude-sensitive complement to total energy and helps assess whether power concentration patterns are stable across the corpus.

In [ ]:
sns.histplot(df_all["rms"], kde=True)
plt.title("Figure 3: Distribution of RMS")
plt.show()

### 3.4 Correlation Structure of Core Features
Correlation analysis identifies potentially redundant predictors and informs feature-selection priorities for classification and calibration workflows.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df_all[cols].corr(), annot=True, cmap="coolwarm")
plt.title("Figure 4: Correlation Heatmap")
plt.show()

### 3.5 Epoch Count by Dataset
This plot checks source balance between CHB-MIT and Siena. Strong imbalance can bias model learning and should be considered during splitting and weighting.

In [ ]:
sns.countplot(x="dataset", data=df_all)
plt.title("Figure 5: Count of Epochs by Dataset")
plt.show()

### 3.6 Channel-Count Composition by Dataset
Channel-count differences can introduce domain shift. This visualization supports interpretation of normalization features such as energy per channel.

In [ ]:
sns.countplot(x="dataset", data=df_all, hue="channel_count")
plt.title("Figure 6: Channel Count by Dataset")
plt.show()

### 3.7 Bivariate Separation: Standard Deviation vs Energy
This scatter view assesses whether datasets exhibit overlapping or distinct feature manifolds, informing transferability of a shared seizure prediction model.

In [ ]:
sns.scatterplot(x="std", y="energy", hue="dataset", data=df_all)
plt.title("Figure 7: Std vs Energy")
plt.show()

### 3.8 Temporal Energy Trend Across Epoch Index
Mean energy over epoch index provides a coarse view of non-stationarity. Drift patterns may motivate temporal validation and sequence-aware modeling.

In [ ]:
sns.lineplot(x="epoch_index", y="energy", data=df_all, estimator="mean")
plt.title("Figure 8: Energy Trend Across Epochs")
plt.show()

## Limitations and Transition to Modeling
This milestone notebook uses one EDF file per dataset to validate preprocessing and feature-engineering decisions.
In the next iteration, the same pipeline will be scaled across additional recordings and paired with seizure interval labeling to support supervised prediction experiments.
The resulting wrangled feature table serves as input to downstream classification, calibration, and cross-dataset evaluation workflows.